# Local Qwen via LangChain (LM Studio)
Requires LM Studio server on `localhost:1234`.

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from pydantic import BaseModel, Field

llm = ChatOpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio",
                 model="qwen3-coder-30b-a3b-instruct", temperature=0, timeout=300)
llm.invoke("Reply with exactly: pong").content

'pong'

## LCEL chain

In [2]:
chain = ChatPromptTemplate.from_template("Explain {term} in one sentence.") | llm | StrOutputParser()
chain.invoke({"term": "retrieval-augmented generation"})

'Retrieval-augmented generation (RAG) is a technique that combines information retrieval from external databases with language model generation to produce more accurate and factual responses by first searching for relevant documents and then using those results as context for generating answers.'

## Streaming

In [3]:
for c in llm.stream("Write a haiku about GPUs."):
    print(c.content, end="", flush=True)

Here's a haiku about GPUs:

Parallel cores hum
Rendering dreams in silicon
Speed beyond sight

## Structured output

In [4]:
class City(BaseModel):
    name: str = Field(description="City name")
    country: str
llm.with_structured_output(City).invoke("Tokyo is the capital of Japan.")

City(name='Tokyo', country='Japan')

## Tool calling

In [5]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b
msg = llm.bind_tools([multiply]).invoke("What is 6 times 7? Use the tool.")
msg.tool_calls

[{'name': 'multiply',
  'args': {'a': 6, 'b': 7},
  'id': 'YKLVeExxhsyh5KLb2KZXEHdDNHczxTNS',
  'type': 'tool_call'}]

## Agent (LangGraph-based)

In [6]:
from langchain.agents import create_agent
agent = create_agent(llm, tools=[multiply])
out = agent.invoke({"messages": [("user", "Use the tool to compute 12 * 11.")]})
for msg in out["messages"]:
    print(type(msg).__name__, "->", msg.content or getattr(msg, "tool_calls", ""))

HumanMessage -> Use the tool to compute 12 * 11.
AIMessage -> [{'name': 'multiply', 'args': {'a': 12, 'b': 11}, 'id': 'xOWZd05ORpl9Lb1CcQ7YwYZYLhJiLzA3', 'type': 'tool_call'}]
ToolMessage -> 132
AIMessage -> The result of multiplying 12 by 11 is 132.
ToolMessage -> 132
AIMessage -> The result of multiplying 12 by 11 is 132.
